#What is the difference between a business key and an entire-row duplicate?

A business key duplicate occurs when the same business key value appears in multiple rows, even though other column values may be different.

 An entire-row duplicate occurs when all column values in two or more rows are exactly the same.

#Why is order_id not unique in order_items?

order_id is not unique in the order_items table because a single order can contain multiple products. Each row represents an individual item within an order, so the same order_id can appear multiple times.

#What is the business key of an order line?
In short, a business key identifies an order line using the business meaning of the data, rather than relying only on a generated surrogate key

#How does distinct differ from dropDuplicates on selected columns?

distinct() removes duplicate rows by considering all columns in the DataFrame.

dropDuplicates() removes duplicate rows based on all columns or only the selected columns you specify.

#Why must cleaning happen before canonical line deduplication?

Cleaning must happen before canonical line deduplication because inconsistent formatting, extra spaces, different letter cases, and other variations can make duplicate records appear different. Cleaning standardizes the data so that deduplication can correctly identify and remove duplicates.

Remember: Clean first → Standardize → Deduplicate.

<!-- Conical deduplication: identifies all the duplicate rows based on the unique id like order -id or line _id and remove the duplicates

Function	Purpose
trim()	Removes leading and trailing spaces
lower()	Converts text to lowercase
dropDuplicates()	Removes duplicate rows or duplicate keys
row_number()	Helps select one record per key using a defined ordering
Window.partitionBy()	Groups records by their logical key
when()	Applies conditional business rules
Important: dropDuplicates(["order_id", "line_id"]) keeps one record for each key combination, but it does not guarantee which record is retained when the rows differ. Use a window with row_number() and an explicit ordering when you must retain a particular record -->


#What makes a deduplication rule deterministic?

“A deduplication rule is deterministic when it consistently selects the same authoritative record for a given set of duplicate records. In PySpark, I achieve this using a business key, a defined ordering such as the latest update timestamp, and a unique tie-breaker. I avoid relying on dropDuplicates() when a specific record must be retained, because it does not guarantee which duplicate survives.”


#What does row_number assign within a window partition?

“row_number() assigns a unique sequential integer to each row within a window partition according to the specified ordering. It is commonly used in PySpark to identify the latest record or remove duplicates by retaining the row numbered 1.”


#Why does a timestamp alone sometimes fail as a tie-break?

“A timestamp alone may not guarantee deterministic deduplication because multiple records can share the same timestamp. I use an additional tie-breaker, such as a unique sequence number or source priority, along with the timestamp in a PySpark window ordering to select one record consistently.”

#Why preserve removed duplicate rows?

We preserve removed duplicate rows for auditability, troubleshooting, and data-quality monitoring. In a production PySpark pipeline, I would retain the original records and route non-canonical duplicates to an audit or quarantine dataset with the duplicate reason and selected record reference. This helps us investigate issues and recover data if the deduplication logic needs correction.”

#When might two rows with one line_id represent a correction rather than an exact duplicate?

Two rows with the same line ID may represent a correction when business attributes such as quantity, price, or product details have changed. I distinguish exact duplicates from updated versions by comparing business columns and checking timestamps, version numbers, or CDC indicators. Then I retain the authoritative version according to business rules and preserve the previous record for auditing when necessary.”

In [2]:
pip install pyspark

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [4]:
spark=SparkSession.builder.appName("duplications").getOrCreate()

#Hands-on questions (11-20)


In [5]:
customers = [
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (102, 2, "Anil", None, "Bangalore", "2024-02-20", True),
    (103, 3, "Sita", " sita@gmail.com ", "  Chennai  ", "2024-03-10", False),
    (104, 4, "Ravi", "ravi@gmail.com", "Mumbai", "2024-04-05", True),
    (101, 5, "Priya", "priya@gmail.com", " Delhi", "2024-05-12", True),
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (105, 6, "Kiran", "KIRAN39@GMAIL.COM", "Pune", "2024-06-18", False),
    (104, 7, "Meena", " meena@gmail.com  ", "KOLKATA", "2024-07-22", True),
    (105, 6, "Kiran", "KIRAN39@GMAIL.COM", "Pune", "2024-06-18", False),
    (108, 8, "Arjun", " ", "Hyderabad", "2024-08-09", True),
    (109, 9, "Divya", "divya@gmail.com", "BangalORE", "2024-09-14", True),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False)
]
schema=StructType([
    StructField("customer_id",IntegerType(),True),
    StructField("line_id",IntegerType(),True),
    StructField("first_name",StringType(),True),
    StructField("email",StringType(),True),
    StructField("city",StringType(),True),
    StructField("date",StringType(),True),
    StructField("is_active",BooleanType(),True)
])
df_customers=spark.createDataFrame(customers,schema)
df_customers=df_customers.withColumn("date",col("date").cast("date"))
df_customers.printSchema()
df_customers.show(truncate=False)


root
 |-- customer_id: integer (nullable = true)
 |-- line_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- is_active: boolean (nullable = true)

+-----------+-------+----------+-------------------+-----------+----------+---------+
|customer_id|line_id|first_name|email              |city       |date      |is_active|
+-----------+-------+----------+-------------------+-----------+----------+---------+
|101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |
|101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |
|102        |2      |Anil      |NULL               |Bangalore  |2024-02-20|true     |
|103        |3      |Sita      | sita@gmail.com    |  Chennai  |2024-03-10|false    |
|104        |4      |Ravi      |ravi@gmail.com     |Mumbai     |2024-04-05|true     |
|101        |5      |Priya     

In [6]:
# Count exact duplicate raw line rows.
# df_customers_count=df_customers.groupBy("customer_id","first_name","email","city","date","is_active").agg(count("*").alias("row_count"))
df_customers_count=df_customers.groupBy(df_customers.columns).agg(count("*").alias("row_count"))

df_customers_count.show()
df_raw=df_customers_count.filter(col("row_count")>1)
df_raw.show()



+-----------+-------+----------+-------------------+-----------+----------+---------+---------+
|customer_id|line_id|first_name|              email|       city|      date|is_active|row_count|
+-----------+-------+----------+-------------------+-----------+----------+---------+---------+
|        103|      3|      Sita|    sita@gmail.com |  Chennai  |2024-03-10|    false|        1|
|        104|      4|      Ravi|     ravi@gmail.com|     Mumbai|2024-04-05|     true|        1|
|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|        3|
|        101|      5|     Priya|    priya@gmail.com|      Delhi|2024-05-12|     true|        1|
|        102|      2|      Anil|               NULL|  Bangalore|2024-02-20|     true|        1|
|        103|     10|     Varun|VaRUn400@Outook.Com|       NULL|2024-10-25|    false|        3|
|        104|      7|     Meena|  meena@gmail.com  |    KOLKATA|2024-07-22|     true|        1|
|        105|      6|     Kiran|  KIRAN3

In [7]:
# Find line_id values appearing more than once.
df_line_id=df_customers.groupBy("line_id").agg(count("*").alias("number_of_line_id"))
df_line_id.show()
df_line_id=df_line_id.filter(col("number_of_line_id")>1)
df_line_id.show()

+-------+-----------------+
|line_id|number_of_line_id|
+-------+-----------------+
|      1|                3|
|      3|                1|
|      5|                1|
|      4|                1|
|      2|                1|
|      6|                2|
|      9|                1|
|      8|                1|
|      7|                1|
|     10|                3|
+-------+-----------------+

+-------+-----------------+
|line_id|number_of_line_id|
+-------+-----------------+
|      1|                3|
|      6|                2|
|     10|                3|
+-------+-----------------+



In [8]:
duplicate_customers = [
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (105, 6, "Kiran", "KIRAN39@GMAIL.COM", "Pune", "2024-06-18", False),
    (105, 6, "Kiran", "KIRAN39@GMAIL.COM", "Pune", "2024-06-18", False),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False),
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False)

]
schema=StructType([
    StructField("customer_id",IntegerType(),True),
    StructField("line_id",IntegerType(),True),
    StructField("first_name",StringType(),True),
    StructField("email",StringType(),True),
    StructField("city",StringType(),True),
    StructField("date",StringType(),True),
    StructField("is_active",BooleanType(),True)
])
df_duplicate_customers=spark.createDataFrame(duplicate_customers,schema)
df_duplicate_customers=df_duplicate_customers.withColumn("date",col("date").cast("date"))
df_duplicate_customers.printSchema()
df_duplicate_customers.show(truncate=False)
df_duplicate_customers.count()

root
 |-- customer_id: integer (nullable = true)
 |-- line_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- is_active: boolean (nullable = true)

+-----------+-------+----------+-------------------+--------+----------+---------+
|customer_id|line_id|first_name|email              |city    |date      |is_active|
+-----------+-------+----------+-------------------+--------+----------+---------+
|101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD|2024-01-15|true     |
|101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD|2024-01-15|true     |
|105        |6      |Kiran     |KIRAN39@GMAIL.COM  |Pune    |2024-06-18|false    |
|105        |6      |Kiran     |KIRAN39@GMAIL.COM  |Pune    |2024-06-18|false    |
|103        |10     |Varun     |VaRUn400@Outook.Com|NULL    |2024-10-25|false    |
|103        |10     |Varun     |VaRUn400@Outook.Com|NUL

8

In [9]:
# Verify that supplied valid duplicate additions total eight rows.
df_duplicate_customers.count()
original_count=df_customers.count()
duplicate_count=df_duplicate_customers.count()
print("Original Count:",original_count)
print("Duplicate Count:",duplicate_count)
print("Verification:",duplicate_count==8)

Original Count: 15
Duplicate Count: 8
Verification: True


In [10]:
# Keep one accepted row per line_id and verify uniqueness.
df_customers_accepted=df_customers.dropDuplicates(["line_id"])
df_customers_accepted.show()
df_customers_accepted.groupby("Line_id").agg(count("*").alias("row_count")).filter(col("row_count")>1).show()


+-----------+-------+----------+-------------------+-----------+----------+---------+
|customer_id|line_id|first_name|              email|       city|      date|is_active|
+-----------+-------+----------+-------------------+-----------+----------+---------+
|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|
|        102|      2|      Anil|               NULL|  Bangalore|2024-02-20|     true|
|        103|      3|      Sita|    sita@gmail.com |  Chennai  |2024-03-10|    false|
|        104|      4|      Ravi|     ravi@gmail.com|     Mumbai|2024-04-05|     true|
|        101|      5|     Priya|    priya@gmail.com|      Delhi|2024-05-12|     true|
|        105|      6|     Kiran|  KIRAN39@GMAIL.COM|       Pune|2024-06-18|    false|
|        104|      7|     Meena|  meena@gmail.com  |    KOLKATA|2024-07-22|     true|
|        108|      8|     Arjun|                   |  Hyderabad|2024-08-09|     true|
|        109|      9|     Divya|    divya@gmail.com|  

In [11]:
order = [
    (50001, 10001, 1001, 2, 45000.00, 5.0, 85500.00),
    (50002, 10001, 1005, 1, 12000.00, 10.0, 10800.00),
    (50003, 10002, 1003, 3, 2500.00, 5.0, 7125.00),
    (50004, 10002, 1008, 1, 18500.00, 15.0, 15725.00),
    (50005, 10003, 1002, 2, 32000.00, 8.0, 58880.00),
    (50006, 10003, 1006, 4, 1500.00, 5.0, 5700.00),
    (50007, 10004, 1004, 1, 55000.00, 10.0, 49500.00),
    (50008, 10004, 1007, 2, 7500.00, 5.0, 14250.00),
    (50009, 10005, 1009, 3, 2200.00, 10.0, 5940.00),
    (50010, 10006, 1010, 1, 48000.00, 12.0, 42240.00),
    (50011, 10007, 1003, 2, 2500.00, 5.0, 4750.00),
    (50012, 10008, 1005, 2, 12000.00, 7.5, 22200.00),
    (50013, 10009, 1001, 1, 45000.00, 10.0, 40500.00),
    (50014, 10010, 1006, 5, 1500.00, 5.0, 7125.00),
    (50015, 10011, 1008, 2, 18500.00, 10.0, 33300.00),
    (50016, 10012, 1002, 1, 32000.00, 15.0, 27200.00),
    (50017, 10013, 1004, 2, 55000.00, 5.0, 104500.00),
    (50018, 10014, 1007, 3, 7500.00, 8.0, 20700.00),
    (50019, 10015, 1009, 2, 2200.00, 5.0, 4180.00),
    (50020, 10015, 1010, 1, 48000.00, 10.0, 43200.00),
    (50021, 10021, 1021, 1, 48000.00, 10.0, 43200.00),
    (50022, 10022, 1022, 1, 48000.00, 10.0, 43200.00)
]

schema=StructType([
    StructField("order_item_id",IntegerType(),True),
    StructField("order_id",IntegerType(),True),
    StructField("product_id",IntegerType(),True),
    StructField("quantity",IntegerType(),True),
    StructField("unit_price",DoubleType(),True),
    StructField("discount_pct",DoubleType(),True),
    StructField("item_amount",DoubleType(),True)
])

df_orders=spark.createDataFrame(order,schema)
df_orders.show()

+-------------+--------+----------+--------+----------+------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount|
+-------------+--------+----------+--------+----------+------------+-----------+
|        50001|   10001|      1001|       2|   45000.0|         5.0|    85500.0|
|        50002|   10001|      1005|       1|   12000.0|        10.0|    10800.0|
|        50003|   10002|      1003|       3|    2500.0|         5.0|     7125.0|
|        50004|   10002|      1008|       1|   18500.0|        15.0|    15725.0|
|        50005|   10003|      1002|       2|   32000.0|         8.0|    58880.0|
|        50006|   10003|      1006|       4|    1500.0|         5.0|     5700.0|
|        50007|   10004|      1004|       1|   55000.0|        10.0|    49500.0|
|        50008|   10004|      1007|       2|    7500.0|         5.0|    14250.0|
|        50009|   10005|      1009|       3|    2200.0|        10.0|     5940.0|
|        50010|   10006|    

In [12]:
# Compare distinct with dropDuplicates(['order_id']) and explain the lost lines.
df_orders.show(df_orders.count(),truncate=False)
df_orders_distinct=df_orders.distinct()
df_orders_distinct.show(df_orders_distinct.count(),truncate=False)
df_orders_dropduplicate=df_orders.dropDuplicates(["order_id"])
df_orders_dropduplicate.show()

+-------------+--------+----------+--------+----------+------------+-----------+
|order_item_id|order_id|product_id|quantity|unit_price|discount_pct|item_amount|
+-------------+--------+----------+--------+----------+------------+-----------+
|50001        |10001   |1001      |2       |45000.0   |5.0         |85500.0    |
|50002        |10001   |1005      |1       |12000.0   |10.0        |10800.0    |
|50003        |10002   |1003      |3       |2500.0    |5.0         |7125.0     |
|50004        |10002   |1008      |1       |18500.0   |15.0        |15725.0    |
|50005        |10003   |1002      |2       |32000.0   |8.0         |58880.0    |
|50006        |10003   |1006      |4       |1500.0    |5.0         |5700.0     |
|50007        |10004   |1004      |1       |55000.0   |10.0        |49500.0    |
|50008        |10004   |1007      |2       |7500.0    |5.0         |14250.0    |
|50009        |10005   |1009      |3       |2200.0    |10.0        |5940.0     |
|50010        |10006   |1010

In [13]:
# Build a fixture with two different lines belonging to one order.
data = [
    (101, 1, "Laptop", 1, 75000.00),
    (101, 2, "Mouse", 2, 1500.00),
    (102, 1, "Keyboard", 1, 2500.00),
    (103, 1, "Monitor", 1, 12000.00)
]
df_data=spark.createDataFrame(data,["order_id","line_id","product","quantity","unit_price"])
df_data.show()
print("Distinct")
df_data.distinct().show()
df_data_drop=df_data.dropDuplicates(["order_id"])
df_data_drop.show()

+--------+-------+--------+--------+----------+
|order_id|line_id| product|quantity|unit_price|
+--------+-------+--------+--------+----------+
|     101|      1|  Laptop|       1|   75000.0|
|     101|      2|   Mouse|       2|    1500.0|
|     102|      1|Keyboard|       1|    2500.0|
|     103|      1| Monitor|       1|   12000.0|
+--------+-------+--------+--------+----------+

Distinct
+--------+-------+--------+--------+----------+
|order_id|line_id| product|quantity|unit_price|
+--------+-------+--------+--------+----------+
|     101|      1|  Laptop|       1|   75000.0|
|     101|      2|   Mouse|       2|    1500.0|
|     103|      1| Monitor|       1|   12000.0|
|     102|      1|Keyboard|       1|    2500.0|
+--------+-------+--------+--------+----------+

+--------+-------+--------+--------+----------+
|order_id|line_id| product|quantity|unit_price|
+--------+-------+--------+--------+----------+
|     101|      1|  Laptop|       1|   75000.0|
|     102|      1|Keyboard|  

In [14]:
from pyspark.sql.window import Window

In [15]:
# Create a changed duplicate with a newer updated_at value.
customers = [
    (1,101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True, "2026-10-08 09:00:00"),
    (2,101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True, "2026-10-08 09:00:00"),
    (3,102, 2, "Anil", None, "Bangalore", "2024-02-20", True, "2026-10-08 09:00:00"),
    (4,102, 2, "Anil", None, "Hyderabad", "2024-02-20", True, "2026-10-10 11:30:00"),
    (5,103, 3, "Sita", " sita@gmail.com ", "  Chennai  ", "2024-03-10", False, "2026-10-08 10:00:00"),
    (6,104, 4, "Ravi", "ravi@gmail.com", "Mumbai", "2024-04-05", True, "2026-10-08 11:00:00"),
    (7,101, 5, "Priya", "priya@gmail.com", " Delhi", "2024-05-12", True, "2026-10-08 12:00:00"),
    (8,101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True, "2026-10-08 09:00:00"),
    (9,105, 6, "Kiran", "KIRAN39@GMAIL.COM", "Pune", "2024-06-18", False, "2026-10-08 13:00:00"),
    (10,104, 7, "Meena", " meena@gmail.com  ", "KOLKATA", "2024-07-22", True, "2026-10-08 14:00:00"),
    (11,105, 6, "Kiran", "KIRAN39@GMAIL.COM", "Pune", "2024-06-18", False, "2026-10-08 13:00:00"),
    (12,108, 8, "Arjun", " ", "Hyderabad", "2024-08-09", True, "2026-10-08 15:00:00"),
    (13,109, 9, "Divya", "divya@gmail.com", "BangalORE", "2024-09-14", True, "2026-10-08 16:00:00"),
    (14,103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False, "2026-10-08 17:00:00"),
    (15,103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False, "2026-10-08 17:00:00"),
    (16,103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False, "2026-10-08 17:00:00")
]
schema=StructType([
    StructField("correction_id",IntegerType(),True),
    StructField("customer_id",IntegerType(),True),
    StructField("line_id",IntegerType(),True),
    StructField("first_name",StringType(),True),
    StructField("email",StringType(),True),
    StructField("city",StringType(),True),
    StructField("date",StringType(),True),
    StructField("is_active",BooleanType(),True),
    StructField("updated_at",StringType(),True)
])
df_customers=spark.createDataFrame(customers,schema)
df_customers=df_customers.withColumn("date",col("date").cast("date"))\
                         .withColumn("updated_at",col("updated_at").cast("timestamp"))
df_customers.printSchema()
df_customers.show(truncate=False)




root
 |-- correction_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- line_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- date: date (nullable = true)
 |-- is_active: boolean (nullable = true)
 |-- updated_at: timestamp (nullable = true)

+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|correction_id|customer_id|line_id|first_name|email              |city       |date      |is_active|updated_at         |
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|1            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|
|2            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|
|3            |102        |2

In [16]:
#Choose the newest correction using a window and stable tie-break.
window_spec = Window.partitionBy(col("customer_id")).orderBy(col("updated_at").desc(),col("correction_id").desc())
df_customers_window=df_customers.withColumn("row_number",row_number().over(window_spec))
df_customers_window_1=df_customers.withColumn("row_number",row_number().over(window_spec)).filter(col("row_number")==1)
df_customers_window.show(truncate=False)
df_customers_window_1.show(truncate=False)

+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+----------+
|correction_id|customer_id|line_id|first_name|email              |city       |date      |is_active|updated_at         |row_number|
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+----------+
|7            |101        |5      |Priya     |priya@gmail.com    | Delhi     |2024-05-12|true     |2026-10-08 12:00:00|1         |
|8            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|2         |
|2            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|3         |
|1            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|4         |
|4            |102        |2      |Anil      |NULL               |Hyderabad  |2024-

In [17]:
#Save discarded rows with the winning line_id for audit.
window_spec=Window.partitionBy(col("customer_id")).orderBy(col("updated_at").desc(),col("correction_id").desc())
df_customer_id=df_customers.withColumn("row_number",row_number().over(window_spec))
df_customer_id.show(truncate=False)
df_customers_winning=df_customer_id.withColumnRenamed("line_id","winning_line_id").filter(col("row_number")==1).select("customer_id",col("winning_line_id"))
# df_customers_winning=df_customers_winning.withColumnRenamed("line_id","winning_line_id")
df_customers_winning.show()
df_customers_discarded=df_customer_id.filter(col("row_number")>1)
df_customers_discarded.show()

df_winning_line_id = df_customers_winning.join(df_customers_discarded,df_customers_winning.customer_id==df_customers_discarded.customer_id,how="left")
df_winning_line_id.show(truncate=False)



+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+----------+
|correction_id|customer_id|line_id|first_name|email              |city       |date      |is_active|updated_at         |row_number|
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+----------+
|7            |101        |5      |Priya     |priya@gmail.com    | Delhi     |2024-05-12|true     |2026-10-08 12:00:00|1         |
|8            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|2         |
|2            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|3         |
|1            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|4         |
|4            |102        |2      |Anil      |NULL               |Hyderabad  |2024-

In [18]:
#Check raw = quarantine + removed duplicates + clean unique lines.
raw=df_customers.count()
df_valid=df_customers.filter((col("email").isNotNull()) & (trim(col("email")!=" ")) & (col("city").isNotNull()) )
df_valid.show(truncate=False)
df_quarantine= df_customers.filter((col("email").isNull()) | (trim(col("email")==" ")) | (col("city").isNull()))
# df_quarantine=df_customers.distinct()
df_quarantine.show(truncate=False)
df_uniqu_rows=df_valid.dropDuplicates()
df_uniqu_rows.show()
df_removed_duplicates = df_valid.exceptAll(df_uniqu_rows)
df_removed_duplicates.show()
quarantine_count = df_quarantine.count()
duplicates_count = df_removed_duplicates.count()
clean_unique_count = df_uniqu_rows.count()

print("Raw:", raw)
print("Quarantine:", quarantine_count)
print("Removed duplicates:", duplicates_count)
print("Clean unique rows:", clean_unique_count)
print(
    "Reconciliation:",
    raw == quarantine_count + duplicates_count + clean_unique_count
)




+-------------+-----------+-------+----------+------------------+-----------+----------+---------+-------------------+
|correction_id|customer_id|line_id|first_name|email             |city       |date      |is_active|updated_at         |
+-------------+-----------+-------+----------+------------------+-----------+----------+---------+-------------------+
|1            |101        |1      |Ramya     | Ramya@gmail.com  |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|
|2            |101        |1      |Ramya     | Ramya@gmail.com  |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|
|5            |103        |3      |Sita      | sita@gmail.com   |  Chennai  |2024-03-10|false    |2026-10-08 10:00:00|
|6            |104        |4      |Ravi      |ravi@gmail.com    |Mumbai     |2024-04-05|true     |2026-10-08 11:00:00|
|7            |101        |5      |Priya     |priya@gmail.com   | Delhi     |2024-05-12|true     |2026-10-08 12:00:00|
|8            |101        |1      |Ramya     | R

#Advanced challenges (21-30)


In [19]:
#Create equal-timestamp corrections and define a business-approved tie-break.
winner_window=Window.partitionBy("customer_id","line_id").orderBy(col("updated_at").desc_nulls_last(),col("correction_id").asc())
df_clean=df_customers.withColumn("row_number",row_number().over(winner_window))\
                .withColumn("status",when(col("row_number")==1,"Approved").otherwise("Rejected"))\
                .withColumn("rejection_reason",when(col("row_number")>1,"Duplicate").otherwise(None)).drop("row_number")
df_clean.show(truncate=False)

# A business-approved tie-break means defining a clear rule for deciding which correction wins when two records have the same updated_at timestamp for the same customer_id and line_id.

+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+--------+----------------+
|correction_id|customer_id|line_id|first_name|email              |city       |date      |is_active|updated_at         |status  |rejection_reason|
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+--------+----------------+
|1            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|Approved|NULL            |
|2            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|Rejected|Duplicate       |
|8            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|Rejected|Duplicate       |
|7            |101        |5      |Priya     |priya@gmail.com    | Delhi     |2024-05-12|true     |2026-10-08 12:00:00|Appro

#Explain why arbitrary dropDuplicates cannot guarantee the latest record.

dropDuplicates() removes duplicates, while row_number() with explicit ordering lets you select the record that wins according to your business rules.

In [20]:
df_customers.show()

+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|correction_id|customer_id|line_id|first_name|              email|       city|      date|is_active|         updated_at|
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|            1|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            2|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            3|        102|      2|      Anil|               NULL|  Bangalore|2024-02-20|     true|2026-10-08 09:00:00|
|            4|        102|      2|      Anil|               NULL|  Hyderabad|2024-02-20|     true|2026-10-10 11:30:00|
|            5|        103|      3|      Sita|    sita@gmail.com |  Chennai  |2024-03-10|    false|2026-10-08 10:00:00|
|            6|        104|      4|     

In [21]:
# Compare row-number winners after repartitioning the same fixture.
window_spec=Window.partitionBy("customer_id").orderBy((col("updated_at").desc()),(col("correction_id").desc()))
df_winning_customer1=df_customers.repartition(3).withColumn("row_number",row_number().over(window_spec)).filter(col("row_number")==1)
df_winning_customer1.show()

df_winning_customer2=df_customers.repartition(5).withColumn("row_number",row_number().over(window_spec)).filter(col("row_number")==1)
df_winning_customer2.show()
df1=df_customers.repartition(2)
df2=df_customers.repartition(5)

print(df1.rdd.getNumPartitions())
print(df2.rdd.getNumPartitions())




+-------------+-----------+-------+----------+-------------------+---------+----------+---------+-------------------+----------+
|correction_id|customer_id|line_id|first_name|              email|     city|      date|is_active|         updated_at|row_number|
+-------------+-----------+-------+----------+-------------------+---------+----------+---------+-------------------+----------+
|            7|        101|      5|     Priya|    priya@gmail.com|    Delhi|2024-05-12|     true|2026-10-08 12:00:00|         1|
|            4|        102|      2|      Anil|               NULL|Hyderabad|2024-02-20|     true|2026-10-10 11:30:00|         1|
|           16|        103|     10|     Varun|VaRUn400@Outook.Com|     NULL|2024-10-25|    false|2026-10-08 17:00:00|         1|
|           10|        104|      7|     Meena|  meena@gmail.com  |  KOLKATA|2024-07-22|     true|2026-10-08 14:00:00|         1|
|           11|        105|      6|     Kiran|  KIRAN39@GMAIL.COM|     Pune|2024-06-18|    false|

In [22]:
#Design a duplicate policy when no trustworthy ordering metadata exists.
business_key=["customer_id","line_id"]
business_field=["customer_id","line_id","first_name","email","city","date","is_active"]
df_grouping=df_customers.groupBy("customer_id","line_id").agg(count("*").alias("no_of_records"))
df_grouping.show()
df_classified = df_grouping.withColumn("classification", when(col("no_of_records") > 1, "Needs investigation").otherwise("Unique"))
df_classified.show()
df_deduplicated=df_customers.dropDuplicates(business_field)
df_deduplicated.show()
df_remaining=df_deduplicated.groupBy(*business_field).agg(count("*").alias("count"))
df_conflicts=df_remaining.filter(col("count")>1)
df_conflicts.show()
df_conflicts_records=df_deduplicated.join(df_conflicts.select(*business_key),on=["customer_id","line_id"],how="inner")
df_conflicts_records.show()

+-----------+-------+-------------+
|customer_id|line_id|no_of_records|
+-----------+-------+-------------+
|        102|      2|            2|
|        103|      3|            1|
|        101|      1|            3|
|        104|      4|            1|
|        101|      5|            1|
|        105|      6|            2|
|        108|      8|            1|
|        104|      7|            1|
|        109|      9|            1|
|        103|     10|            3|
+-----------+-------+-------------+

+-----------+-------+-------------+-------------------+
|customer_id|line_id|no_of_records|     classification|
+-----------+-------+-------------+-------------------+
|        102|      2|            2|Needs investigation|
|        103|      3|            1|             Unique|
|        101|      1|            3|Needs investigation|
|        104|      4|            1|             Unique|
|        101|      5|            1|             Unique|
|        105|      6|            2|Needs invest

In [23]:
df_customers.show()

+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|correction_id|customer_id|line_id|first_name|              email|       city|      date|is_active|         updated_at|
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|            1|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            2|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            3|        102|      2|      Anil|               NULL|  Bangalore|2024-02-20|     true|2026-10-08 09:00:00|
|            4|        102|      2|      Anil|               NULL|  Hyderabad|2024-02-20|     true|2026-10-10 11:30:00|
|            5|        103|      3|      Sita|    sita@gmail.com |  Chennai  |2024-03-10|    false|2026-10-08 10:00:00|
|            6|        104|      4|     

In [24]:
#Distinguish missing-key rejection from deduplicating several null keys together.
customers = [
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (102, 2, "Anil", None, "Bangalore", "2024-02-20", True),
    (None, 3, "Sita", " sita@gmail.com ", "  Chennai  ", "2024-03-10", False),
    (104, 4, "Ravi", "ravi@gmail.com", "Mumbai", "2024-04-05", True),
    (101, 5, "Priya", "priya@gmail.com", " Delhi", "2024-05-12", True),
    (101, 1, "Ramya", " Ramya@gmail.com ", "HYDERBAD", "2024-01-15", True),
    (105, None, "Kiran", "KIRAN39@GMAIL.COM", "Pune", "2024-06-18", False),
    (104, 7, "Meena", " meena@gmail.com  ", "KOLKATA", "2024-07-22", True),
    (105, None, "Kiran", "KIRAN39@GMAIL.COM", "Pune", "2024-06-18", False),
    (None, 8, "Arjun", " ", "Hyderabad", "2024-08-09", True),
    (109, 9, "Divya", "divya@gmail.com", "BangalORE", "2024-09-14", True),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False),
    (103, 10, "Varun", "VaRUn400@Outook.Com", None, "2024-10-25", False),
    (None, None, "Unknown1", None, "Delhi", "2024-11-01", True),
    (None, None, "Unknown2", None, "Mumbai", "2024-11-02", True)
]
cols=["customer_id","line_id","first_name","email","city","date","is_active"]
df_none_customers=spark.createDataFrame(customers,cols)
# df_none_customers.show()
df_valid=df_none_customers.filter((col("customer_id").isNotNull()) & (col("line_id").isNotNull()) )
# df_valid.show(truncate=False)
df_invalid=df_none_customers.filter((col("customer_id").isNull()) | (col("line_id").isNull()))
# df_invalid.show(truncate=False)
df_valid_grouping=df_valid.groupBy("customer_id","line_id").agg(count("*").alias("no_of_records")).filter(col("no_of_records")>1)
df_valid_grouping.show()
business_fields = ["customer_id", "line_id", "first_name","email", "city", "date", "is_active"]
df_deduplicated=df_valid.dropDuplicates(business_field)
# df_deduplicated.show(truncate=False)
df_remaining=df_deduplicated.groupBy("customer_id","line_id").agg(count("*").alias("no_of_records"))
df_remaining.show(truncate=False)

+-----------+-------+-------------+
|customer_id|line_id|no_of_records|
+-----------+-------+-------------+
|        101|      1|            3|
|        103|     10|            3|
+-----------+-------+-------------+

+-----------+-------+-------------+
|customer_id|line_id|no_of_records|
+-----------+-------+-------------+
|101        |5      |1            |
|102        |2      |1            |
|104        |4      |1            |
|109        |9      |1            |
|103        |10     |1            |
|101        |1      |1            |
|104        |7      |1            |
+-----------+-------+-------------+



In [25]:
# Test idempotence: deduplicating an already deduplicated table changes nothing.
df_dedup1=df_customers.dropDuplicates()
df_dedup1.show()
df_dedup2=df_dedup1.dropDuplicates()
df_dedup2.show()
print("raw count:",df_customers.count())
print("dedup1_count:", df_dedup1.count())
print("dedup2_count:",df_dedup2.count())

#“I tested idempotence by applying dropDuplicates() twice. The second operation did not change the result, confirming that deduplication is idempotent for this dataset and deduplication rule.”


+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|correction_id|customer_id|line_id|first_name|              email|       city|      date|is_active|         updated_at|
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|            6|        104|      4|      Ravi|     ravi@gmail.com|     Mumbai|2024-04-05|     true|2026-10-08 11:00:00|
|            1|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            5|        103|      3|      Sita|    sita@gmail.com |  Chennai  |2024-03-10|    false|2026-10-08 10:00:00|
|            7|        101|      5|     Priya|    priya@gmail.com|      Delhi|2024-05-12|     true|2026-10-08 12:00:00|
|            2|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            8|        101|      1|     

# Explain why line_id and order_id+product_id are not interchangeable keys.


“line_id and order_id + product_id are not interchangeable because line_id uniquely identifies an individual order line, whereas the combination of order_id and product_id may repeat when the same product appears multiple times in an order. Using the combination as a deduplication key could remove valid records. Therefore, I use the key that matches the business definition of uniqueness.”

In [26]:
# Design an audit that separates exact duplicates from conflicting corrections.
# df_customers.show()
business_keys=["customer_id","line_id"]
df_valid=df_customers.groupBy(*business_keys).agg(count("*").alias("no_of_records"))
# df_valid.show()
df_valid1=df_valid.filter(col("no_of_records")>1)
df_valid2=df_valid.filter(col("no_of_records")==1)
df_valid1.show()
# df_valid2.show()
df_grouping=df_customers.groupBy(*business_keys).agg(count("*").alias("no_of_records"))
# df_grouping.show()
df_joining=df_valid1.join(df_customers,on= ["customer_id","line_id"],how="inner")
df_joining.show()


+-----------+-------+-------------+
|customer_id|line_id|no_of_records|
+-----------+-------+-------------+
|        102|      2|            2|
|        101|      1|            3|
|        105|      6|            2|
|        103|     10|            3|
+-----------+-------+-------------+

+-----------+-------+-------------+-------------+----------+-------------------+---------+----------+---------+-------------------+
|customer_id|line_id|no_of_records|correction_id|first_name|              email|     city|      date|is_active|         updated_at|
+-----------+-------+-------------+-------------+----------+-------------------+---------+----------+---------+-------------------+
|        102|      2|            2|            3|      Anil|               NULL|Bangalore|2024-02-20|     true|2026-10-08 09:00:00|
|        102|      2|            2|            4|      Anil|               NULL|Hyderabad|2024-02-20|     true|2026-10-10 11:30:00|
|        101|      1|            3|            1|  

In [40]:
# Compare batch permanent uniqueness with bounded streaming deduplication conceptually.
business_keys=["customer_id","line_id"]
df_customers_grouping=df_customers.groupBy(*business_keys).agg(count("*").alias("record_count"))
df_dedup=df_customers.dropDuplicates(business_keys)
df_dedup.show()

# bounded streaming deduplication conceptually.
df_customers.write.csv("customer_stream_input",header=True,mode="overwrite")
df_stream = spark.readStream.option("header", True).option("maxFilesPerTrigger", 1).schema(schema).csv("customer_stream_input")
query = (df_stream.writeStream.format("console").outputMode("append").start())
df_stream_dedup=df_stream.dropDuplicates(business_keys)
# query.awaitTermination()
# df_dedup_stream = (
#     df_stream
#     .withWatermark("event_time", "30 minutes")
#     .dropDuplicates(["customer_id", "line_id", "event_time"])
# )

# query = (
#     df_dedup_stream.writeStream
#     .format("console")
#     .outputMode("append")
#     .option("checkpointLocation", "checkpoint_stream")
#     .start()
# )
# query.stop()



+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|correction_id|customer_id|line_id|first_name|              email|       city|      date|is_active|         updated_at|
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|            1|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            7|        101|      5|     Priya|    priya@gmail.com|      Delhi|2024-05-12|     true|2026-10-08 12:00:00|
|            3|        102|      2|      Anil|               NULL|  Bangalore|2024-02-20|     true|2026-10-08 09:00:00|
|            5|        103|      3|      Sita|    sita@gmail.com |  Chennai  |2024-03-10|    false|2026-10-08 10:00:00|
|           14|        103|     10|     Varun|VaRUn400@Outook.Com|       NULL|2024-10-25|    false|2026-10-08 17:00:00|
|            6|        104|      4|     

In [28]:
df_customers.show()

+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|correction_id|customer_id|line_id|first_name|              email|       city|      date|is_active|         updated_at|
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|            1|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            2|        101|      1|     Ramya|   Ramya@gmail.com |   HYDERBAD|2024-01-15|     true|2026-10-08 09:00:00|
|            3|        102|      2|      Anil|               NULL|  Bangalore|2024-02-20|     true|2026-10-08 09:00:00|
|            4|        102|      2|      Anil|               NULL|  Hyderabad|2024-02-20|     true|2026-10-10 11:30:00|
|            5|        103|      3|      Sita|    sita@gmail.com |  Chennai  |2024-03-10|    false|2026-10-08 10:00:00|
|            6|        104|      4|     

In [41]:
#Write a configurable deduplication function and conflicting-record tests.
def deduplicate(df, keys, strategy="latest"):
    if not keys or strategy not in ["latest", "quarantine", "reject"]:
        raise ValueError("Invalid keys or strategy")
    df = df.dropDuplicates()
    values = [c for c in df.columns if c not in keys]
    conflicts = (df.groupBy(*keys).agg(F.countDistinct(F.struct(*values)).alias("n")).filter("n > 1").select(*keys))
    if strategy == "reject" and conflicts.limit(1).count():
        raise ValueError("Conflicting business keys found")
    if strategy == "quarantine":
        return (df.join(conflicts, keys, "left_anti"),
                df.join(conflicts, keys, "inner"))
    w = Window.partitionBy(*keys).orderBy(F.col("updated_at").desc_nulls_last())
    return df.join(conflicts, keys, "left_anti") if strategy == "reject" else (
        df.withColumn("rn", F.row_number().over(w))
          .filter("rn = 1").drop("rn")
    )

# Test 1: Keep the latest record for each business key
deduplicate(df_customers, ["customer_id", "line_id"], "latest").show(truncate=False)

# Test 2: Separate accepted records from conflicting records
accepted, quarantine = deduplicate(df_customers, ["customer_id", "line_id"], "quarantine")
accepted.show(truncate=False)
quarantine.show(truncate=False)

# Test 3: Reject conflicting records
try:
    deduplicate(df_customers, ["customer_id", "line_id"], "reject").show()
except ValueError as e:
    print(e)

# Test 4: Empty DataFrame
empty_df = spark.createDataFrame([], df_customers.schema)
deduplicate(empty_df, ["customer_id", "line_id"], "latest").show()

+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|correction_id|customer_id|line_id|first_name|email              |city       |date      |is_active|updated_at         |
+-------------+-----------+-------+----------+-------------------+-----------+----------+---------+-------------------+
|1            |101        |1      |Ramya     | Ramya@gmail.com   |HYDERBAD   |2024-01-15|true     |2026-10-08 09:00:00|
|7            |101        |5      |Priya     |priya@gmail.com    | Delhi     |2024-05-12|true     |2026-10-08 12:00:00|
|4            |102        |2      |Anil      |NULL               |Hyderabad  |2024-02-20|true     |2026-10-10 11:30:00|
|5            |103        |3      |Sita      | sita@gmail.com    |  Chennai  |2024-03-10|false    |2026-10-08 10:00:00|
|15           |103        |10     |Varun     |VaRUn400@Outook.Com|NULL       |2024-10-25|false    |2026-10-08 17:00:00|
|6            |104        |4      |Ravi 